In [1]:
%pip install -q cairosvg

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from pathlib import Path
import json
import subprocess
import sys
import cairosvg
import numpy as np
import cv2

from IPython.display import Image, SVG, display

In [5]:
PROJECT_ROOT = Path.cwd().parents[1]

SCRIPT = (
  PROJECT_ROOT / "01_scripts" / "test_generate.py"
)

OUTPUT_DIR = PROJECT_ROOT / "02_notebooks" / "generation" / "test_outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Script:", SCRIPT)
print("Output directory:", OUTPUT_DIR)

Project root: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter
Script: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\01_scripts\test_generate.py
Output directory: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\02_notebooks\generation\test_outputs


In [6]:
command = [
  sys.executable, str(SCRIPT),
  "--output-dir", str(OUTPUT_DIR),
  "--canvas-px", "25",
  "--target-visible-px", "15",
  "--rotation", "0",
  "--render-png",
]

result = subprocess.run(
  command,
  capture_output=True,
  text=True,
  check=True,
)

print(result.stdout)

{
  "svg_path": "c:\\Users\\xxxAn\\Desktop\\projects\\crochet-pattern-converter\\02_notebooks\\generation\\test_outputs\\sc_square_cross_canvas25_rot0.svg",
  "png_path": "c:\\Users\\xxxAn\\Desktop\\projects\\crochet-pattern-converter\\02_notebooks\\generation\\test_outputs\\sc_square_cross_canvas25_rot0.png",
  "metadata": {
    "class_id": 0,
    "class_name": "sc",
    "phenotype": "square_symmetric_cross",
    "canvas": {
      "width_px": 25,
      "height_px": 25
    },
    "target_visible_px": 15.0,
    "estimated_visible_width_px": 15.0,
    "estimated_visible_height_px": 15.0,
    "viewbox": [
      0,
      0,
      100,
      100
    ],
    "visual_rotation_deg": 0.0,
    "obb_angle_deg": 0.0,
    "orientation_policy": "canonical",
    "stroke_width_normalized": 4.0,
    "geometry": {
      "horizontal_arm": {
        "x1": 22,
        "y1": 50,
        "x2": 78,
        "y2": 50
      },
      "vertical_arm": {
        "x1": 50,
        "y1": 22,
        "x2": 50,
        "

In [8]:
svg_path = next(OUTPUT_DIR.glob("*.svg"))
png_path = next(OUTPUT_DIR.glob("*.png"))
json_path = next(OUTPUT_DIR.glob("*.json"))
label_path = next(OUTPUT_DIR.glob("*.txt"))

print("SVG:", svg_path)
print("PNG:", png_path)
print("JSON:", json_path)
print("Label:", label_path)

SVG: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\02_notebooks\generation\test_outputs\sc_square_cross_canvas25_rot0.svg
PNG: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\02_notebooks\generation\test_outputs\sc_square_cross_canvas25_rot0.png
JSON: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\02_notebooks\generation\test_outputs\sc_square_cross_canvas25_rot0.json
Label: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\02_notebooks\generation\test_outputs\sc_square_cross_canvas25_rot0.txt


In [9]:
metadata = json.loads(
  json_path.read_text(encoding="utf-8")
)

print(json.dumps(metadata, indent=2))

print("\nYOLO OBB label:")
print(label_path.read_text(encoding="utf-8"))

{
  "class_id": 0,
  "class_name": "sc",
  "phenotype": "square_symmetric_cross",
  "canvas": {
    "width_px": 25,
    "height_px": 25
  },
  "target_visible_px": 15.0,
  "estimated_visible_width_px": 15.0,
  "estimated_visible_height_px": 15.0,
  "viewbox": [
    0,
    0,
    100,
    100
  ],
  "visual_rotation_deg": 0.0,
  "obb_angle_deg": 0.0,
  "orientation_policy": "canonical",
  "stroke_width_normalized": 4.0,
  "geometry": {
    "horizontal_arm": {
      "x1": 22,
      "y1": 50,
      "x2": 78,
      "y2": 50
    },
    "vertical_arm": {
      "x1": 50,
      "y1": 22,
      "x2": 50,
      "y2": 78
    },
    "approx_visible_bounds_normalized": [
      20,
      20,
      80,
      80
    ]
  },
  "obb": {
    "pixels": [
      [
        5.0,
        5.0
      ],
      [
        20.0,
        5.0
      ],
      [
        20.0,
        20.0
      ],
      [
        5.0,
        20.0
      ]
    ],
    "normalized": [
      [
        0.2,
        0.2
      ],
      [
        

In [11]:
display(SVG(filename=str(svg_path)))
display(Image(filename=str(png_path)))

In [15]:
def draw_obb_overlay(
  image_path: Path,
  obb_pixels: np.ndarray,
  output_path: Path,
  color: tuple[int, int, int] = (0, 0, 255),
  thickness: int = 2,
) -> None:
  image = cv2.imread(str(image_path))

  if image is None:
    raise FileNotFoundError(
      f"Could not read image: {image_path}"
    )

  points = np.round(
    obb_pixels
  ).astype(np.int32)

  polygon = points.reshape((-1, 1, 2))

  cv2.polylines(
    image,
    [polygon],
    isClosed=True,
    color=color,
    thickness=thickness,
    lineType=cv2.LINE_AA,
  )

  # Draw corner indices for debugging.
  for index, point in enumerate(obb_pixels):
    x, y = np.round(point).astype(int)

    cv2.circle(
      image,
      (x, y),
      radius=4,
      color=(255, 0, 0),
      thickness=-1,
    )

    cv2.putText(
      image,
      str(index),
      (x + 6, y - 6),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.6,
      (255, 0, 0),
      2,
      cv2.LINE_AA,
    )

  output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
  )

  success = cv2.imwrite(
    str(output_path),
    image,
  )

  if not success:
    raise RuntimeError(
      f"Failed to write overlay: {output_path}"
    )

In [14]:
obb_pixels = np.asarray(
  metadata["obb"]["pixels"],
  dtype=np.float32,
)

print("OBB pixel coordinates:")
print(obb_pixels)

OBB pixel coordinates:
[[ 5.  5.]
 [20.  5.]
 [20. 20.]
 [ 5. 20.]]


In [18]:
overlay_path = OUTPUT_DIR / (
  f"{png_path.stem}_overlay.png"
)

draw_obb_overlay(
  image_path=png_path,
  obb_pixels=obb_pixels,
  output_path=overlay_path,
)

print("Overlay:", overlay_path)

display(Image(filename=str(overlay_path)))

Overlay: c:\Users\xxxAn\Desktop\projects\crochet-pattern-converter\02_notebooks\generation\test_outputs\sc_square_cross_canvas25_rot0_overlay.png


In [19]:
from IPython.display import HTML, display

html = f"""
<div style="display:flex; gap:24px; align-items:flex-start;">
  <div>
    <h4>OBB overlay</h4>
    <img src="{overlay_path}" style="image-rendering: pixelated;">
  </div>
  <div>
    <h4>Metadata</h4>
    <pre>{json.dumps(metadata, indent=2)}</pre>
  </div>
</div>
"""

display(HTML(html))

In [20]:
label_values = label_path.read_text(
  encoding="utf-8"
).strip().split()

class_id_from_txt = int(label_values[0])

coords_from_txt = np.asarray(
  [float(value) for value in label_values[1:]],
  dtype=np.float32,
).reshape(4, 2)

reconstructed_pixels = coords_from_txt.copy()

reconstructed_pixels[:, 0] *= canvas_px
reconstructed_pixels[:, 1] *= canvas_px

print("Class ID:", class_id_from_txt)
print("Reconstructed pixels:")
print(reconstructed_pixels)

assert class_id_from_txt == metadata["class_id"]

np.testing.assert_allclose(
  reconstructed_pixels,
  obb_pixels,
  atol=1e-4,
)

print("YOLO label round-trip passed.")

NameError: name 'canvas_px' is not defined

In [10]:
LARGE_OUTPUT_DIR = OUTPUT_DIR / "magnified"
LARGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

command = [
  sys.executable, str(SCRIPT),
  "--output-dir", str(LARGE_OUTPUT_DIR),
  "--canvas-px", "250",
  "--target-visible-px", "150",
  "--rotation", "0",
  "--render-png",
]

subprocess.run(command, check=True)

large_png = (
  LARGE_OUTPUT_DIR
  / "sc_square_cross_canvas250_rot0.png"
)

In [ ]:
display(Image(filename=str(large_png)))

In [ ]:
ROTATION_DIR = OUTPUT_DIR / "rotation_sweep"
ROTATION_DIR.mkdir(parents=True, exist_ok=True)

angles = [0, 15, 30, 45, 60, 75, 90, 135]

generated = []

for angle in angles:
  command = [
    sys.executable, str(SCRIPT),
    "--output-dir", str(ROTATION_DIR),
    "--canvas-px", "250",
    "--target-visible-px", "150", 
    "--rotation", str(angle),
    "--render-png",
  ]

  subprocess.run(command, check=True)

  png_path = (
    ROTATION_DIR
    / f"sc_square_cross_canvas250_rot{angle:g}.png"
  )

  generated.append({
    "angle": angle,
    "png_path": png_path,
  })

generated

In [ ]:
for item in generated:
  print(f"Rotation: {item['angle']}°")
  display(Image(filename=str(item["png_path"])))

In [ ]:
metadata = json.loads(json_path.read_text())

assert metadata["class_name"] == "sc"
assert metadata["phenotype"] == "square_symmetric_cross"

assert metadata["canvas_px"] == 25
assert metadata["target_visible_px"] == 15.0

assert metadata["geometry"]["approx_visible_bounds_normalized"] == [
  20,
  20,
  80,
  80
]